# MODULE 2 — Logique de programmation, NumPy et simulation
**Semaines 3–4 • 6 heures en classe**

## Question directrice
**Pouvez-vous traduire une décision en logique claire, puis appliquer le calcul à grande échelle avec des tableaux NumPy ?**

*Cours conçu par le Professeur BOISGUENE RUBBENS*

### Explication du module
Passer de valeurs isolées à des règles, boucles, fonctions et tableaux numériques pour automatiser des calculs contrôlables, puis utiliser la simulation pour voir la variabilité due au hasard.

### Contenu du module
- *Semaine 3* — Conditions if / elif / else
- *Semaine 3* — Boucles for / while, enumerate et zip
- *Semaine 3* — Fonctions, paramètres et valeurs de retour
- *Semaine 3* — Fichiers, textes bruts et exceptions (try / except)
- *Semaine 4* — NumPy : arrays, formes, types et indexation
- *Semaine 4* — Masques booléens, vectorisation et broadcasting
- *Semaine 4* — Statistiques descriptives avec NumPy
- *Semaine 4* — Générateur aléatoire et simulation : la variabilité d’échantillonnage

### Outils
Python 3, datetime, NumPy (dont le générateur aléatoire), pandas

### Routine de classe
1. Discuter de la tâche pendant 10 à 20 minutes avant de coder.
2. Prédire les sorties avant d’exécuter les démonstrations.
3. Utiliser au moins deux contrôles de validation pertinents.
4. Redémarrer le noyau et exécuter toutes les cellules avant la remise.
5. Conclure par : conclusion, preuves, limite, étape suivante.

### Assistants d’IA
Autorisés pour comprendre et explorer, jamais pour remplacer votre raisonnement. Déclarez tout usage à la fin du notebook.

> **VERSION ÉTUDIANTE** — discutez d’abord. N’ouvrez le corrigé qu’après avoir consigné votre raisonnement et votre tentative de code.

In [ ]:
# Path : manipuler des chemins de fichiers de façon portable (Windows, Mac, Colab)
from pathlib import Path
# NumPy : calcul numérique sur des tableaux
import numpy as np
# pandas : tableaux de données (DataFrame)
import pandas as pd
# Matplotlib : tracer des graphiques
import matplotlib.pyplot as plt
# Seaborn : graphiques statistiques (boxplot, heatmap…) construits sur Matplotlib
import seaborn as sns
# display : afficher proprement un tableau dans le notebook
from IPython.display import display
# datetime : lire des dates et heures ; timedelta : représenter une durée (ex. 3 heures)
from datetime import datetime, timedelta
# Afficher jusqu'à 30 colonnes quand un tableau est large
pd.set_option("display.max_columns", 30)
# Générateur aléatoire avec graine fixe (42) : mêmes résultats à chaque exécution
rng = np.random.default_rng(42)

# Dossiers où chercher les données, dans l'ordre (à côté du notebook, dossier parent, dossier courant)
DOSSIERS_DONNEES = [Path("09_Jeux_de_Donnees_Synthetiques"), Path("../09_Jeux_de_Donnees_Synthetiques"), Path(".")]

# Fonction qui trouve un fichier de données, quel que soit l'endroit où le notebook est ouvert
def chemin_donnees(nom_fichier):
    """Renvoie le chemin d’un fichier du dossier 09_Jeux_de_Donnees_Synthetiques."""
    # On essaie chaque dossier candidat, l'un après l'autre
    for dossier in DOSSIERS_DONNEES:
        # On construit le chemin complet : dossier / nom du fichier
        chemin = dossier / nom_fichier
        # Si le fichier existe à cet endroit…
        if chemin.exists():
            # … on renvoie ce chemin et la fonction s'arrête
            return chemin
    # Si aucun dossier ne contient le fichier, on arrête avec un message qui explique quoi faire
    raise FileNotFoundError(
        f"« {nom_fichier} » introuvable. Dans Colab : téléversez le fichier (icône dossier à gauche). "  # message, partie 1
        "En local : gardez le dossier 09_Jeux_de_Donnees_Synthetiques à côté de ce notebook.")  # message, partie 2

# Message de confirmation : si vous le voyez, la cellule s'est exécutée sans erreur
print("Environnement prêt — données synthétiques, graine 42.")

---
# Semaine 3 — Rappel Python II : décisions, boucles, fonctions, fichiers et textes

---
# Scénario 4 — Validation des expéditions de secours [CAS PRINCIPAL]
*Semaine 3* · Données : `S04_expeditions_secours.csv` (synthétiques)

### Contexte
Après la tempête, des partenaires humanitaires envoient vers le Sud de l’eau, du riz, des bâches et des kits d’hygiène. Chaque expédition est saisie par un agent logistique avec son identifiant, sa quantité, le poids unitaire et une case indiquant si le chargement a été vérifié sur le terrain. Le tonnage total déclaré sert à planifier les camions et à rendre des comptes aux bailleurs : une erreur de saisie peut donc coûter cher. Avant de consolider les chiffres, vous devez traduire les règles de validation en une fonction qui accepte, corrige ou bloque chaque ligne.

## PAUSE — Discutez de la tâche analytique avant de toucher au clavier
Travaillez en binôme ou en petit groupe pendant **10 à 20 minutes**. Consignez les six réponses avant de coder.

**1. Quelle décision pourrait évoluer ?**  
_Votre réponse :_ _______________________________________________

**2. Qui est la partie prenante ?**  
_Votre réponse :_ _______________________________________________

**3. Que représente une ligne ?**  
_Votre réponse :_ _______________________________________________

**4. Quels champs et quelles unités sont nécessaires ?**  
_Votre réponse :_ _______________________________________________

**5. À quoi doit ressembler le résultat ?**  
_Votre réponse :_ _______________________________________________

**6. De quelles façons la réponse pourrait-elle être erronée ?**  
_Votre réponse :_ _______________________________________________

### Avant de coder, mettez-vous d’accord sur
- l’unité d’analyse et le grain d’une ligne ;
- les champs requis et leurs unités ;
- au moins deux contrôles de validation ;
- la forme du résultat final attendu.

> ⚠️ **Ce jeu de données contient au moins un piège propre à ce scénario.** Trouvez-le et expliquez-le avant de conclure.

In [ ]:
# Lire le fichier des expéditions de secours
expeditions = pd.read_csv(chemin_donnees("S04_expeditions_secours.csv"))
# Afficher toutes les lignes (repérez les doublons et les valeurs étranges)
display(expeditions)

### Étapes
1. Écrire la règle métier en français : quand accepter, corriger, bloquer ?
2. Traduire cette règle dans une fonction statut_expedition(...) qui renvoie (statut, motif).
3. Parcourir les lignes avec une boucle ; repérer une expédition déjà vue.
4. Calculer le tonnage naïf (toutes lignes) puis le tonnage des seules lignes acceptées.
5. Vérifier l’unicité de shipment_id parmi les lignes acceptées et recalculer une ligne à la main.

*Indice : Idée : garder un `set` des identifiants déjà vus ; `pd.isna(valeur)` teste une valeur manquante.*

In [ ]:
# À FAIRE — scénario 4 : écrivez votre code ci-dessous, en suivant les étapes.
# Gardez chaque transformation visible et ajoutez vos deux contrôles avec assert.


# AFFICHAGE OBLIGATOIRE DE LA TABLE FINALE
# display(resultat)

### Après le code — réflexion obligatoire
**Conclusion :**
**Preuves (avec unités) :**
**Limite :**
**Étape suivante :**

**Contrôle de validation 1 :**
**Contrôle de validation 2 :**
**Piège trouvé dans les données :**

### Note à la partie prenante — à rédiger après avoir inspecté votre table finale
Rédigez une **note de 120 à 180 mots**. Ne décrivez pas Python : expliquez ce que le résultat signifie pour la décision. Elle doit contenir une recommandation, **au moins deux résultats chiffrés avec leurs unités**, leur interprétation, une limite et une étape suivante.

_À : Responsables logistiques et partenaires humanitaires_
_Objet : Validation des expéditions de secours — point analytique_

__________________________________________________________________

---
# Scénario 6 — Analyse des SMS d’urgence [MISSION EXPRESS]
*Semaine 3* · Données : `S06_sms_urgence.txt` (synthétiques)

### Contexte
Le 3 septembre 2026 au matin, la coordination de terrain reçoit des SMS de relais communautaires du Sud : un besoin en eau à Torbeck, une route bloquée à Chantal, des abris détruits à Maniche. Chaque message suit en principe le format « LIEU|THÈME|STATUT|NOMBRE|HORODATAGE », mais il est tapé vite, sur un vieux téléphone et parfois sous la pluie. La coordinatrice n’a pas le temps de tout lire à la main : elle a besoin d’une liste propre d’événements et du nombre de personnes concernées par thème. Votre programme doit lire le fichier texte ligne par ligne, sans s’arrêter à la première ligne mal formée.

## PAUSE — Discutez de la tâche analytique avant de toucher au clavier
Travaillez en binôme ou en petit groupe pendant **10 à 20 minutes**. Consignez les six réponses avant de coder.

**1. Quelle décision pourrait évoluer ?**  
_Votre réponse :_ _______________________________________________

**2. Qui est la partie prenante ?**  
_Votre réponse :_ _______________________________________________

**3. Que représente une ligne ?**  
_Votre réponse :_ _______________________________________________

**4. Quels champs et quelles unités sont nécessaires ?**  
_Votre réponse :_ _______________________________________________

**5. À quoi doit ressembler le résultat ?**  
_Votre réponse :_ _______________________________________________

**6. De quelles façons la réponse pourrait-elle être erronée ?**  
_Votre réponse :_ _______________________________________________

### Avant de coder, mettez-vous d’accord sur
- l’unité d’analyse et le grain d’une ligne ;
- les champs requis et leurs unités ;
- au moins deux contrôles de validation ;
- la forme du résultat final attendu.

> ⚠️ **Ce jeu de données contient au moins un piège propre à ce scénario.** Trouvez-le et expliquez-le avant de conclure.

In [ ]:
# Lire le fichier texte, retirer les espaces de fin, puis découper en une liste de lignes
lignes = chemin_donnees("S06_sms_urgence.txt").read_text(encoding="utf-8").strip().splitlines()
# La première ligne est l'en-tête ; les suivantes sont les SMS
entete, messages = lignes[0], lignes[1:]
# Afficher le nombre de SMS et l'en-tête
print(f"{len(messages)} SMS lus. En-tête : {entete}")
# Afficher chaque SMS brut, un par ligne
for m in messages:
    # afficher le SMS courant
    print(m)

### Étapes
1. Écrire une fonction lire_sms(texte) qui découpe un SMS et lève ValueError s’il est invalide.
2. Parcourir les SMS avec try / except : conserver les valides, journaliser les rejets avec leur motif.
3. Vérifier : SMS lus = valides + rejetés.
4. Définir une règle de dédoublonnage (même lieu, thème, statut, nombre dans une fenêtre de 3 h).
5. Comparer le total de personnes par thème avant et après dédoublonnage.

*Indice : Fonctions utiles : `texte.split("|")`, `.strip()`, `.title()`, `str.isdigit()`, `datetime.strptime(..., "%Y-%m-%d %H:%M")`.*

In [ ]:
# À FAIRE — scénario 6 : écrivez votre code ci-dessous, en suivant les étapes.
# Gardez chaque transformation visible et ajoutez vos deux contrôles avec assert.


# AFFICHAGE OBLIGATOIRE DE LA TABLE FINALE
# display(resultat)

### Après le code — réflexion obligatoire
**Conclusion :**
**Preuves (avec unités) :**
**Limite :**
**Étape suivante :**

**Contrôle de validation 1 :**
**Contrôle de validation 2 :**
**Piège trouvé dans les données :**

### Note à la partie prenante — à rédiger après avoir inspecté votre table finale
Rédigez une **note de 120 à 180 mots**. Ne décrivez pas Python : expliquez ce que le résultat signifie pour la décision. Elle doit contenir une recommandation, **au moins deux résultats chiffrés avec leurs unités**, leur interprétation, une limite et une étape suivante.

_À : Coordinateur ou coordinatrice de terrain_
_Objet : Analyse des SMS d’urgence — point analytique_

__________________________________________________________________

---
## Semaine 3 — Récapitulatif : qu’avons-nous appris aujourd’hui ?
- Transformer une règle métier en condition explicite.
- Utiliser boucles et fonctions sans dupliquer inutilement le code.
- Lire des fichiers et des textes bruts en prévoyant les erreurs et en documentant les hypothèses.

### Rappel actif — sans exécuter de code
1. **Un concept que je peux expliquer :** __________________________________________
2. **Un contrôle que j’utiliserais :** _____________________________________________
3. **Une erreur que je sais maintenant éviter :** _______________________________

---
# Semaine 4 — NumPy : raisonner avec des tableaux et simuler le hasard

## Concept — Simuler le hasard : la variabilité d’échantillonnage
Une enquête n’interroge qu’un **échantillon**. Deux échantillons différents donnent deux estimations différentes. Avec NumPy, on peut **simuler** ce phénomène : on crée une population synthétique dont on connaît la vraie valeur, puis on tire des milliers d’échantillons.

**Prédiction avant exécution :** si l’on passe de 100 à 400 ménages interrogés, l’intervalle qui contient 95 % des estimations sera-t-il 4 fois, 2 fois ou 1,5 fois plus étroit ?

In [ ]:
# Population synthétique : 5 000 ménages, chacun servi avec une probabilité de 70 % (True = servi)
population = rng.random(5000) < 0.70

# Fonction qui simule une enquête de taille donnée
def estimer(taille_echantillon):
    """Tire un échantillon sans remise et renvoie la proportion de ménages servis."""
    # Tirer des ménages au hasard (sans remise) et calculer la part de True
    return rng.choice(population, size=taille_echantillon, replace=False).mean()

# Dictionnaire vide : il contiendra les estimations pour chaque taille d'échantillon
resultats = {}
# Trois tailles d'enquête à comparer
for n in [25, 100, 400]:
    # Simuler 2 000 enquêtes de taille n
    estimations = np.array([estimer(n) for _ in range(2000)])
    # Ranger ces estimations sous la clé n
    resultats[n] = estimations
    # Intervalle qui contient 95 % des estimations (percentiles 2,5 et 97,5)
    bas, haut = np.percentile(estimations, [2.5, 97.5])
    # Afficher l'intervalle et sa largeur ({n:>3} aligne le nombre à droite sur 3 caractères)
    print(f"n = {n:>3} : 95 % des estimations entre {bas:.1%} et {haut:.1%} (largeur {haut - bas:.1%})")

# Créer la figure
fig, ax = plt.subplots(figsize=(7, 3.5))
# Deux tailles à superposer, chacune avec sa couleur
for n, couleur in zip([25, 400], ["#9ecae1", "#1f9e89"]):
    # Histogramme des 2 000 estimations pour cette taille
    ax.hist(resultats[n], bins=30, alpha=0.7, color=couleur, label=f"n = {n}")
# Ligne verticale : la vraie proportion dans la population
ax.axvline(population.mean(), color="black", linestyle="--", label="vraie valeur")
# Titre de l'axe horizontal
ax.set_xlabel("Proportion estimée de ménages servis")
# Titre de l'axe vertical
ax.set_ylabel("Nombre d’échantillons simulés")
# Titre du graphique
ax.set_title("2 000 enquêtes simulées : la taille de l’échantillon réduit la dispersion")
# Légende
ax.legend()
# Ajuster les marges
plt.tight_layout()
# Afficher le graphique
plt.show()

**À retenir :** la dispersion diminue comme 1/√n. Multiplier l’échantillon par 4 divise la largeur par 2. C’est l’idée de l’**erreur standard**, que le module 4 transformera en intervalle de confiance.

---
# Scénario 5 — Matrice des précipitations et de l’exposition aux inondations [CAS PRINCIPAL]
*Semaine 4* · Données : `S05_pluie_7jours.csv` (synthétiques)

### Contexte
Une semaine de fortes pluies a touché six communes côtières, de Gonaïves à Jacmel. Des stations météorologiques ont enregistré la pluie tombée chaque jour, et une enquête indique quelle part des ménages vit en zone inondable. L’équipe de réduction des risques ne peut pas envoyer d’inspecteurs partout en même temps : elle doit choisir où vérifier en premier les digues, les ravines et les abris. Vous travaillerez avec NumPy sur un tableau de chiffres, où chaque ligne est une commune et chaque colonne un jour.

## PAUSE — Discutez de la tâche analytique avant de toucher au clavier
Travaillez en binôme ou en petit groupe pendant **10 à 20 minutes**. Consignez les six réponses avant de coder.

**1. Quelle décision pourrait évoluer ?**  
_Votre réponse :_ _______________________________________________

**2. Qui est la partie prenante ?**  
_Votre réponse :_ _______________________________________________

**3. Que représente une ligne ?**  
_Votre réponse :_ _______________________________________________

**4. Quels champs et quelles unités sont nécessaires ?**  
_Votre réponse :_ _______________________________________________

**5. À quoi doit ressembler le résultat ?**  
_Votre réponse :_ _______________________________________________

**6. De quelles façons la réponse pourrait-elle être erronée ?**  
_Votre réponse :_ _______________________________________________

### Avant de coder, mettez-vous d’accord sur
- l’unité d’analyse et le grain d’une ligne ;
- les champs requis et leurs unités ;
- au moins deux contrôles de validation ;
- la forme du résultat final attendu.

> ⚠️ **Ce jeu de données contient au moins un piège propre à ce scénario.** Trouvez-le et expliquez-le avant de conclure.

In [ ]:
# Lire le fichier des précipitations sur 7 jours
pluie_df = pd.read_csv(chemin_donnees("S05_pluie_7jours.csv"))
# Afficher le tableau (une ligne par commune, une colonne par jour)
display(pluie_df)

### Étapes
1. Construire un tableau NumPy (6 communes × 7 jours) ; afficher forme, type, minimum.
2. Prédire puis calculer le cumul naïf par commune. Quelle valeur est absurde ?
3. Remplacer la valeur sentinelle par NaN grâce à un masque booléen.
4. Calculer cumul observé, cumul estimé sur 7 jours, maximum journalier et indice d’exposition.
5. Signaler une vérification prioritaire si cumul > 300 mm ou maximum journalier > 100 mm.

*Indice : Fonctions utiles : `.to_numpy(dtype=float)`, `tableau == -999`, `np.nan`, `np.nansum`, `np.nanmax`, `axis=1`.*

In [ ]:
# À FAIRE — scénario 5 : écrivez votre code ci-dessous, en suivant les étapes.
# Gardez chaque transformation visible et ajoutez vos deux contrôles avec assert.


# AFFICHAGE OBLIGATOIRE DE LA TABLE FINALE
# display(resultat)

### Après le code — réflexion obligatoire
**Conclusion :**
**Preuves (avec unités) :**
**Limite :**
**Étape suivante :**

**Contrôle de validation 1 :**
**Contrôle de validation 2 :**
**Piège trouvé dans les données :**

### Note à la partie prenante — à rédiger après avoir inspecté votre table finale
Rédigez une **note de 120 à 180 mots**. Ne décrivez pas Python : expliquez ce que le résultat signifie pour la décision. Elle doit contenir une recommandation, **au moins deux résultats chiffrés avec leurs unités**, leur interprétation, une limite et une étape suivante.

_À : Équipe de réduction des risques et autorités communales_
_Objet : Matrice des précipitations et de l’exposition aux inondations — point analytique_

__________________________________________________________________

---
## Semaine 4 — Récapitulatif : qu’avons-nous appris aujourd’hui ?
- Construire, indexer et filtrer un tableau NumPy.
- Remplacer des calculs manuels par une opération vectorisée.
- Simuler des échantillons pour observer la variabilité d’une estimation.

### Rappel actif — sans exécuter de code
1. **Un concept que je peux expliquer :** __________________________________________
2. **Un contrôle que j’utiliserais :** _____________________________________________
3. **Une erreur que je sais maintenant éviter :** _______________________________

---
# Vérification de fin de module
Avant la remise :
- [ ] J’ai répondu aux questions analytiques avant de coder.
- [ ] Je peux expliquer l’unité d’analyse et les unités de mesure essentielles.
- [ ] J’ai trouvé et expliqué le piège de chaque scénario.
- [ ] J’ai utilisé au moins deux contrôles de validation pertinents.
- [ ] J’ai indiqué que les données pédagogiques sont synthétiques.
- [ ] J’ai redémarré le noyau et exécuté toutes les cellules dans l’ordre.
- [ ] Mon résultat final présente une conclusion, des preuves, une limite et une étape suivante.
- [ ] J’ai rempli la déclaration d’usage de l’IA.

## Déclaration d’usage de l’IA (obligatoire)
| Question | Votre réponse |
|---|---|
| Avez-vous utilisé un assistant d’IA ? Lequel ? | |
| Pour quelle(s) tâche(s) précise(s) ? | |
| Qu’avez-vous gardé, modifié ou rejeté ? | |
| Comment avez-vous vérifié ? | |
| Avez-vous transmis des données ? Lesquelles ? | |

## Ticket de sortie du module
Répondez en quelques phrases, sans code :
1. **Quel concept pouvez-vous désormais expliquer sans code ?**
   *Ce que la question signifie :* montrez que vous comprenez l’idée, pas seulement la syntaxe.
2. **Quel contrôle permettrait de détecter une erreur grave dans le scénario d’aujourd’hui ?**
   *Ce que la question signifie :* proposez une vérification concrète capable d’empêcher une conclusion erronée.
3. **Quelle incertitude ou limite subsiste ?**
   *Ce que la question signifie :* précisez ce que les données ou la méthode ne permettent pas d’affirmer.